<a href="https://colab.research.google.com/github/Haruki-24/Challenge-Alura-Agente-OCI/blob/main/notebooks/Challenge_Alura_Agente_RAG_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto: Agente de Operaciones y Control Industrial (OCI)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 - Configuracion de entorno

### 1.1 - Instalación de depedencias

In [2]:
# 1. Creamos el archivo requirements.txt incluyendo el paquete faltante
requirements_content = """
faiss-cpu
google-generativeai
langchain
langchain-community
langchain-core
langchain-google-genai
langgraph
openpyxl
pandas
PyMuPDF
pypdf
python-dotenv
unstructured
langchain-experimental
langchain-huggingface
sentence-transformers
transformers
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

print("Archivo 'requirements.txt' actualizado actualizado con éxito.")

# 2. Instalamos de manera limpia
!pip install -q -r requirements.txt
!pip install -q "unstructured[pdf]"

print("¡Entorno configurado con éxito!")

Archivo 'requirements.txt' actualizado actualizado con éxito.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

### 1.2 - Importación de bibliotecas

In [3]:
import os
from getpass import getpass
import pandas as pd
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

print("Bibliotecas base importadas con éxito.")

Bibliotecas base importadas con éxito.


/tmp/ipykernel_1210/930589896.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 1.3 - Conexión de Modelos LLM

In [4]:
import os
import pandas as pd
from google.colab import userdata


# Definicion de modelos
# Opciones de LLM_PROVIDER: "google", "openai", "cohere"
LLM_PROVIDER = "google"
LLM_MODEL_NAME = "gemini-2.5-flash"  # Ej: "gpt-4o-mini" para OpenAI, "command-r-plus" para Cohere

# Opciones de EMBEDDING_PROVIDER: "huggingface", "google", "openai", "cohere"
EMBEDDING_PROVIDER = "huggingface"
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"  # Ej: "text-embedding-3-small" para OpenAI, "models/text-embedding-004" para Google


# Configuración y recuperación de API Keys de forma dinámica según el proveedor seleccionado
def cargar_api_key(secret_name, env_var_name):
    try:
        key = userdata.get(secret_name)
        os.environ[env_var_name] = key
        return key
    except Exception:
        print(f"No se encontró la credencial '{secret_name}' en los secretos de Colab.")
        return None

# Inicialización dinámica del LLM según el proveedor elegido
print(f"Configurando LLM: {LLM_PROVIDER.upper()} ({LLM_MODEL_NAME})...")

if LLM_PROVIDER == "google":
    cargar_api_key('Gemini-GC', 'GOOGLE_API_KEY')
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=LLM_MODEL_NAME, temperature=0.1)

elif LLM_PROVIDER == "openai":
    cargar_api_key('OPENAI_API_KEY', 'OPENAI_API_KEY')
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=LLM_MODEL_NAME, temperature=0.1)

elif LLM_PROVIDER == "cohere":
    cargar_api_key('COHERE_API_KEY', 'COHERE_API_KEY')
    from langchain_cohere import ChatCohere
    llm = ChatCohere(model=LLM_MODEL_NAME, temperature=0.1)

else:
    raise ValueError(f"Proveedor de LLM no soportado: {LLM_PROVIDER}")

print("Modelo LLM configurado correctamente.")

Configurando LLM: GOOGLE (gemini-2.5-flash)...
Modelo LLM configurado correctamente.


## 2 - Tratamiento y transformacion de datos

### 2.1 - Instalacion unstructured[pdf]

In [5]:
!pip show unstructured # 2. Verificación de la instalación

Name: unstructured
Version: 0.24.0
Summary: A library that prepares raw documents for downstream ML tasks.
Home-page: https://github.com/Unstructured-IO/unstructured
Author: 
Author-email: Unstructured Technologies <devops@unstructuredai.io>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: beautifulsoup4, charset-normalizer, emoji, filelock, filetype, html5lib, installer, langdetect, lxml, numba, numpy, psutil, python-iso639, python-magic, python-oxmsg, rapidfuzz, regex, requests, spacy, tqdm, typing-extensions, unstructured-client, wrapt
Required-by: 


### 2.2 - Cargadores de documentos

In [6]:
# Importaciones necesarias para el cargador fusionado
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, UnstructuredExcelLoader
from langchain_community.document_loaders.merge import MergedDataLoader

# 1. Definición de la ruta de documentos
url_docs = '/content/drive/MyDrive/Colab Notebooks/Challenge Alura 2026/Challenge-Alura-Agente-OCI/data'

# 2. Configuración de cargadores
loader_pdfs = DirectoryLoader(url_docs, glob="*.pdf", loader_cls=PyPDFLoader)
loader_excel = DirectoryLoader(url_docs, glob="*.xlsx", loader_cls=UnstructuredExcelLoader)

# 3.Combinacion de Loaders
all_loaders = MergedDataLoader(loaders=[loader_pdfs, loader_excel])

print("Cargando documentos desde Drive...")
try:
    documentos_oci = all_loaders.load()
    print(f"Se han cargado exitosamente {len(documentos_oci)} documentos en total.")
    if len(documentos_oci) > 0:
        print(f"Primer documento cargado: {documentos_oci[0].metadata.get('source', 'Desconocido')}")
except Exception as e:
    print(f"Error al cargar los archivos: {e}")
    print("Verifica que tu Google Drive esté montado correctamente con: from google.colab import drive; drive.mount('/content/drive')")


Cargando documentos desde Drive...
Se han cargado exitosamente 7 documentos en total.
Primer documento cargado: /content/drive/MyDrive/Colab Notebooks/Challenge Alura 2026/Challenge-Alura-Agente-OCI/data/Politica_HSE_OCI.pdf


In [7]:
# documentos_oci


### 2.3 - Extracción y fragmetación

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print(f"Configurando tokenizador de fragmentación adaptado a {EMBEDDING_PROVIDER.upper()}...")

# 1. Selección dinámica del Splitter de acuerdo al proveedor de Embeddings
if EMBEDDING_PROVIDER == "huggingface":
    from transformers import AutoTokenizer
    print(f"Descargando tokenizador local para '{EMBEDDING_MODEL_NAME}'...")
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=tokenizer,
        chunk_size=400,    # Límite seguro para dar margen a metadatos
        chunk_overlap=50
    )
else:
    # Usamos un codificador estándar por tokens para APIs de nube
    print("Iniciando codificador basado en tiktoken...")
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=400,
        chunk_overlap=50
    )

# 2. Ejecutamos la fragmentación de los documentos cargados
documentos_fragmentados = text_splitter.split_documents(documentos_oci)

# 3. Inyección CONDICIONAL del prefijo "passage: "
# El prefijo es necesario únicamente si el modelo configurado pertenece a la familia E5 de HuggingFace
REQUIERE_PREFIJO_E5 = (EMBEDDING_PROVIDER == "huggingface" and "e5" in EMBEDDING_MODEL_NAME.lower())

if REQUIERE_PREFIJO_E5:
    print("Modelo de la familia E5 detectado. Aplicando prefijo 'passage: ' para indexación...")
    for doc in documentos_fragmentados:
        if not doc.page_content.startswith("passage: "):
            doc.page_content = f"passage: {doc.page_content}"
else:
    print("Modelo estándar detectado. No se requieren prefijos en los fragmentos de almacenamiento.")

print(f"\nFragmentación completada con éxito.")
print(f"Documentos originales: {len(documentos_oci)}")
print(f"Fragmentos (chunks) resultantes listos para indexar: {len(documentos_fragmentados)}")

# Muestra del primer fragmento formateado
if len(documentos_fragmentados) > 0:
    print("\n--- Ejemplo de fragmento ---")
    print(documentos_fragmentados[0].page_content[:300] + "...")


Configurando tokenizador de fragmentación adaptado a HUGGINGFACE...
Descargando tokenizador local para 'intfloat/multilingual-e5-small'...


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Modelo de la familia E5 detectado. Aplicando prefijo 'passage: ' para indexación...

Fragmentación completada con éxito.
Documentos originales: 7
Fragmentos (chunks) resultantes listos para indexar: 11

--- Ejemplo de fragmento ---
passage: Política de Salud, Seguridad y Medio Ambiente (HSE) - OCI
1. Compromiso Fundamental
En Operations & Control Industrial (OCI), reconocemos que la integridad de nuestro personal, la protección dela salud y el cuidado del medio ambiente son valores innegociables. Ninguna tarea operativa es tan...


### 2.4 - Vectorización (Embedings)

In [9]:
from langchain_community.vectorstores import FAISS

print(f"Inicializando modelo de Embeddings: {EMBEDDING_PROVIDER.upper()}...")

# 1. Inicialización dinámica del Embedding Class
if EMBEDDING_PROVIDER == "huggingface":
    from langchain_huggingface import HuggingFaceEmbeddings
    model_kwargs = {'device': 'cpu'}  # Cambiar a 'cuda' si tienes una GPU activa en Colab
    encode_kwargs = {'normalize_embeddings': True}
    embeddings_local = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )

elif EMBEDDING_PROVIDER == "google":
    cargar_api_key('Gemini-GC', 'GOOGLE_API_KEY')
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embeddings_local = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME)

elif EMBEDDING_PROVIDER == "openai":
    cargar_api_key('OPENAI_API_KEY', 'OPENAI_API_KEY')
    from langchain_openai import OpenAIEmbeddings
    embeddings_local = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

elif EMBEDDING_PROVIDER == "cohere":
    cargar_api_key('COHERE_API_KEY', 'COHERE_API_KEY')
    from langchain_cohere import CohereEmbeddings
    embeddings_local = CohereEmbeddings(model=EMBEDDING_MODEL_NAME)

else:
    raise ValueError(f"Proveedor de Embeddings no soportado: {EMBEDDING_PROVIDER}")

# 2. Ruta de persistencia local en Colab
FAISS_DB_PATH = "/content/faiss_index_oci"

print("Generando base de datos vectorial FAISS...")
try:
    # Creamos el índice utilizando los fragmentos procesados y la instancia de embeddings configurada
    db = FAISS.from_documents(documentos_fragmentados, embeddings_local)

    # Guardamos localmente el índice estructurado
    db.save_local(FAISS_DB_PATH)
    print(f"¡Base de datos FAISS guardada con éxito en '{FAISS_DB_PATH}'!")
except Exception as e:
    print(f"Error durante el proceso de indexación: {e}")


Inicializando modelo de Embeddings: HUGGINGFACE...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Generando base de datos vectorial FAISS...
¡Base de datos FAISS guardada con éxito en '/content/faiss_index_oci'!


### 2.5 - Recuperacion RAG

In [10]:
def test_de_recuperacion_e5(query, k=3):
    global db # Declare db as global at the start of the function

    """
    Realiza una búsqueda semántica usando el formato asimétrico de E5 ("query: ").
    Retorna una lista de tuplas (Documento, Score) para que el pipeline del LLM pueda usarlo.
    """
    # Formateamos obligatoriamente la consulta para el modelo E5
    query_formateada = f"query: {query}"

    print(f"\nBuscando contexto técnico para: '{query}'")
    print(f"(Consulta formateada enviada a FAISS: '{query_formateada}')")
    print("-" * 75)

    try:
        # Búsqueda por similitud coseno
        resultados = db.similarity_search_with_relevance_scores(query_formateada, k=k)
    except NameError:
        # En caso de que se haya reiniciado el entorno, intentamos cargar el índice desde disco
        print("Variable 'db' no encontrada en memoria. Cargando índice guardado...")
        db = FAISS.load_local(FAISS_DB_PATH, embeddings_local, allow_dangerous_deserialization=True)
        resultados = db.similarity_search_with_relevance_scores(query_formateada, k=k)

    if not resultados:
        print("No se encontraron coincidencias en la base de datos.")
        return []

    for idx, (doc, score) in enumerate(resultados, 1):
        origen = doc.metadata.get('source', 'Desconocido').split('/')[-1]
        print(f"[{idx}] Coincidencia (Relevancia: {score:.4f})")
        print(f"    Fuente original: {origen}")

        # Removemos temporalmente "passage: " solo para facilitar la visualización en pantalla
        contenido_limpio = doc.page_content.replace("passage: ", "").replace('\n', ' ')[:200]
        print(f"    Contenido: {contenido_limpio}...\n")

    return resultados


In [11]:
# --- PRUEBA DE FUNCIONAMIENTO ---
# Probamos con la consulta operativa real
contexto_recuperado = test_de_recuperacion_e5(
    query="tengo que realizar mantenimiento electrico en campo, ¿que debo llevar?",
    k=4
)


Buscando contexto técnico para: 'tengo que realizar mantenimiento electrico en campo, ¿que debo llevar?'
(Consulta formateada enviada a FAISS: 'query: tengo que realizar mantenimiento electrico en campo, ¿que debo llevar?')
---------------------------------------------------------------------------
[1] Coincidencia (Relevancia: 0.8199)
    Fuente original: Procedimiento_y_protocolos_HSE_OCI.pdf
    Contenido: Equipamiento: Uso estricto de elementos certificados: soga, grilletes, eslingas y cadenas. 4. Herramientas y Equipamiento Técnico El personal debe portar la caja de herramientas estándar, complementad...

[2] Coincidencia (Relevancia: 0.8062)
    Fuente original: Procedimiento_y_protocolos_HSE_OCI.pdf
    Contenido: Procedimiento de Trabajo en Campo y Protocolos de EPP - OCI 1. Planificación y Logística Programación: El supervisor debe programar las tareas operativas, idealmente, el día anterior a la ejecución. S...

[3] Coincidencia (Relevancia: 0.7853)
    Fuente original: Prog

## 3 - Desarrollo de Triaje Operativo

### 3.1 - Lógica de Triaje Operativo (Pydantic + Gemini)

In [12]:
from typing import Literal, Dict
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage

# 1. Definición del esquema estructurado con validación estricta
class TriajeOperativoOut(BaseModel):
    decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ALERTAR_SUPERVISOR"] = Field(
        description="Ruta de decisión recomendada para la solicitud operativa."
    )
    urgency: Literal["BAJA", "MEDIANA", "ALTA"] = Field(
        description="Nivel de prioridad o criticidad del escenario planteado."
    )
    motivo: str = Field(
        description="Breve justificación técnica de la categorización asignada."
    )


In [13]:
# 2. Prompt del Agente de Triaje de Operaciones
PROMPT_TRIAJE_OPERATIVO = """
Eres el Agente Inteligente de Triaje Operativo y de Riesgos en Campo (OCI).
Tu tarea es analizar el mensaje del usuario y clasificarlo técnicamente.

Reglas de Decisión (`decision`):
- "AUTO_RESOLVER": Consultas generales sobre manuales, políticas de la empresa, definiciones técnicas estándar o procedimientos de bajo riesgo que no requieran aprobación.
- "PEDIR_INFO": Solicitudes que carecen de datos operativos indispensables (ej. no especifican el pozo, la herramienta exacta, o el componente del que hablan).
- "ALERTAR_SUPERVISOR": Situaciones críticas de seguridad (HSE), maniobras de alto riesgo (como trabajos en altura o pulling), reporte de averías críticas en pozos, o solicitudes que exijan validación de firma y control humano.

Reglas de Urgencia (`urgency`):
- "BAJA": Consultas generales sin impacto directo en la producción de hoy (ej. políticas de vacaciones en campo, consulta conceptual de un equipo).
- "MEDIANA": Planificación de mantenimiento preventivo rutinario, reposición de herramientas menores o control de stock planificado.
- "ALTA": Emergencias de seguridad (HSE), parada imprevista de un pozo, cronogramas de pulling retrasados o falta de EPP crítico para una maniobra inmediata.

Devuelve SOLO el objeto JSON estructurado según el esquema solicitado.
"""

In [14]:
# Vinculación del LLM para obtener JSON estructurado
chain_de_triaje_operativo = llm.with_structured_output(TriajeOperativoOut)

def ejecutar_triaje_operativo(mensaje: str) -> Dict:
    """Invoca al LLM para clasificar de manera segura la consulta de operaciones."""
    salida: TriajeOperativoOut = chain_de_triaje_operativo.invoke([
        SystemMessage(content=PROMPT_TRIAJE_OPERATIVO),
        HumanMessage(content=mensaje)
    ])
    return salida.model_dump()

# Prueba unitaria de triaje operativo
test_triaje = ejecutar_triaje_operativo("Estamos listos para iniciar pulling en el pozo PCP-04 pero hay tormenta eléctrica")
print("Prueba de triaje operativo exitosa:", test_triaje)


Prueba de triaje operativo exitosa: {'decision': 'ALERTAR_SUPERVISOR', 'urgency': 'ALTA', 'motivo': 'La tormenta eléctrica durante una operación de pulling representa una situación crítica de seguridad (HSE) y una maniobra de alto riesgo que requiere la intervención y validación de un supervisor.'}


In [15]:
!pip install -q langchain-classic

In [16]:
from sentence_transformers import CrossEncoder
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from pydantic import BaseModel, Field

# 1. Inicialización del Reranker local
print("Inicializando modelo de Reranking (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)


# 2. Recuperador semántico básico de FAISS (Celda 5)
def buscar_candidatos_vectoriales(query: str, k: int = 8) -> list:
    """Realiza la búsqueda semántica inicial y devuelve un set amplio de candidatos."""
    # Aplicamos el prefijo 'query: ' únicamente si el modelo de embedding actual requiere protocolo E5
    query_formateada = f"query: {query}" if REQUIERE_PREFIJO_E5 else query

    try:
        global db
        if 'db' not in globals():
            db = FAISS.load_local(FAISS_DB_PATH, embeddings_local, allow_dangerous_deserialization=True)
        return db.similarity_search(query_formateada, k=k)
    except Exception as e:
        print(f"Error al recuperar contexto vectorial: {e}")
        return []



Inicializando modelo de Reranking (cross-encoder/ms-marco-MiniLM-L-6-v2)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### 3.4 - Reclasificación (Reranking)

In [17]:
# 3. Reclasificación (Reranking) de Documentos
def ejecutar_reranking(query: str, documentos: list, top_n: int = 3) -> list:
    """Reordena los fragmentos utilizando un modelo Cross-Encoder para medir relevancia real."""
    if not documentos:
        return []

    # Preparamos los pares (query, contenido_del_documento) para el Cross-Encoder
    # Eliminamos el prefijo "passage: " para que el reranker evalúe el texto limpio
    pares = [[query, doc.page_content.replace("passage: ", "").strip()] for doc in documentos]

    # Calculamos scores de relevancia de forma local en Colab
    scores = reranker_model.predict(pares)

    # Emparejamos documentos con sus respectivos scores y ordenamos descendentemente
    documentos_con_score = list(zip(documentos, scores))
    documentos_ordenados = sorted(documentos_con_score, key=lambda x: x[1], reverse=True)

    # Retornamos los Top N documentos más relevantes
    return [doc for doc, score in documentos_ordenados[:top_n]]

### 3.5 - Ensamblaje del contexto

In [18]:
# 4. Ensamblaje del Contexto Estructurado con Citación de Fuentes Dinámicas
def ensamblar_contexto(documentos_reranked: list) -> tuple:
    """
    Construye un bloque de texto estructurado para el LLM y extrae metadatos limpios de citación.
    Soporta extracción de número de página en PDFs de forma dinámica.
    """
    if not documentos_reranked:
        return "No hay información contextual disponible.", []

    contexto_bloque = ""
    citaciones_limpias = []

    for idx, doc in enumerate(documentos_reranked, 1):
        origen = doc.metadata.get('source', 'Desconocido').split('/')[-1]
        pagina = doc.metadata.get('page', None)

        # Si la fuente es un PDF, registramos el número real de página (0-indexed en PyPDFLoader)
        citacion_label = f"{origen} (Pág. {pagina + 1})" if pagina is not None else origen
        citaciones_limpias.append(citacion_label)

        contenido = doc.page_content.replace("passage: ", "").strip()
        contexto_bloque += f"--- [RECURSO {idx} | Fuente: {citacion_label}] ---\n{contenido}\n\n"

    return contexto_bloque, citaciones_limpias



### 3.6 - Prompt Recuperacion de Triaje

In [19]:
# 5. Prompt del Especialista Técnico de Operaciones (Respetando el Human-in-the-Loop)
prompt_rag_operativo = ChatPromptTemplate([
    ("system",
     """
     Eres el Ingeniero de Soporte Operativo y Mentor de Seguridad (HSE) de la empresa.
     Tu rol es responder la consulta utilizando únicamente el contexto documental provisto.

     REGLAS DE RESPUESTA:
     1. Utiliza un tono profesional, claro y orientado a la seguridad industrial.
     2. Si la consulta involucra procedimientos críticos de mantenimiento o campo, añade siempre un apartado de: "⚠️ ADVERTENCIA DE SEGURIDAD (HSE)".
     3. Si en el contexto técnico proporcionado NO figura la información para responder, contesta estrictamente: "No cuento con registros técnicos suficientes en mis manuales para resolver esta consulta de forma segura."
     4. Finaliza tus recomendaciones técnicas recordando al usuario que el Supervisor de Campo tiene la decisión final frente a cualquier imprevisto técnico o climático.
     """),
    ("human", "Contexto técnico recuperado:\n{context}\n\nPregunta del operador en campo: {input}")
])

document_chain_operativa = create_stuff_documents_chain(llm, prompt=prompt_rag_operativo)


### 3.7 - Validación de respuestas y control de alucinación

In [20]:
# 6. Esquema de validación para el Control de Alucinaciones
class EvaluacionFidelidad(BaseModel):
    fiel_al_contexto: bool = Field(
        description="True si la respuesta se basa ÚNICAMENTE en el contexto técnico proporcionado. False si introduce hechos externos, suposiciones o contradice los documentos."
    )
    justificacion: str = Field(
        description="Justificación detallada de por qué la respuesta se considera fiel o por qué contiene alucinaciones."
    )

evaluador_alucinaciones = llm.with_structured_output(EvaluacionFidelidad)


In [21]:
# 7. Ejecutor del Filtro de Alucinaciones
def validar_respuesta_frente_al_contexto(pregunta: str, contexto: str, respuesta: str) -> EvaluacionFidelidad:
    """Verifica mediante LLM estructurado si la respuesta contiene alucinaciones o datos fuera del contexto."""
    prompt_evaluacion = f"""
    Analiza rigurosamente si la 'Respuesta Propuesta' incurre en alucinaciones basadas en la información provista en el 'Contexto Técnico'.

    [Contexto Técnico]
    {contexto}

    [Pregunta del Operador]
    {pregunta}

    [Respuesta Propuesta]
    {respuesta}

    Devuelve la evaluación estructurada según el esquema solicitado.
    """
    return evaluador_alucinaciones.invoke(prompt_evaluacion)

In [22]:
# 8. Función Unificada de Resolución con RAG Avanzado y Guardrails
def resolver_con_rag_avanzado(pregunta: str) -> Dict:
    """Busca, reclasifica, ensambla contexto, genera respuesta, evalúa alucinaciones y aplica fallback."""
    # A. Recuperación y Reranking
    candidatos = buscar_candidatos_vectoriales(pregunta, k=8)
    documentos_filtrados = ejecutar_reranking(pregunta, candidatos, top_n=3)

    if not documentos_filtrados:
        return {
            "respuesta": "No cuento con registros técnicos suficientes en mis manuales para resolver esta consulta de forma segura.",
            "citaciones": [],
            "rag_exito": False,
            "motivo_fallo": "Búsqueda vectorial vacía."
        }

    # B. Ensamblaje del contexto (contexto_final se usa para el evaluador, no para la cadena)
    contexto_final, citaciones = ensamblar_contexto(documentos_filtrados)

    # C. Generación Inicial de Respuesta
    # IMPORTANTE: Pasamos la lista de objetos Document directamente
    respuesta_ia = document_chain_operativa.invoke({
        "input": pregunta,
        "context": documentos_filtrados
    })

    # Check manual de Fallback explícito
    if "no cuento con registros técnicos" in respuesta_ia.lower():
        return {
            "respuesta": respuesta_ia,
            "citaciones": [],
            "rag_exito": False,
            "motivo_fallo": "Ausencia de coincidencia técnica en manuales."
        }

    # D. Guardrail de Alucinaciones
    evaluacion = validar_respuesta_frente_al_contexto(pregunta, contexto_final, respuesta_ia)

    if not evaluacion.fiel_al_contexto:
        print(f"[GUARDRAIL] Alucinación detectada: {evaluacion.justificacion}")
        return {
            "respuesta": "No cuento con registros técnicos suficientes en mis manuales para resolver esta consulta de forma segura.",
            "citaciones": [],
            "rag_exito": False,
            "motivo_fallo": f"Alucinación detectada: {evaluacion.justificacion}"
        }

    return {
        "respuesta": respuesta_ia,
        "citaciones": citaciones,
        "rag_exito": True
    }

## 4 - Estructura del Grafo en LangGraph (HITL & Orquestación)

### 4.1 - Definicion de AgentState

In [23]:
from typing import TypedDict, Optional, List, Dict
from langgraph.graph import START, END, StateGraph

# 1. Definición del estado de la memoria compartida
class AgentState(TypedDict):
    pregunta: str
    triaje: Dict
    respuesta_RAG: Optional[str]
    citaciones: Optional[List]
    rag_exito: bool
    accion_final: str

### 4.2 - Definicion de nodos Triaje

In [24]:
# 2. Nodo de Triaje Inicial
def nodo_triaje_operaciones(state: AgentState) -> AgentState:
    print("[NODO: Triaje] Analizando riesgos de la consulta...")
    resultado_triaje = ejecutar_triaje_operativo(state["pregunta"])
    return {"triaje": resultado_triaje}

# 3. Nodo Auto Resolver (Consulta RAG Avanzado de baja criticidad)
def nodo_auto_resolver_operaciones(state: AgentState) -> AgentState:
    print("[NODO: Auto Resolver] Consultando manuales técnicos...")
    resultado_rag = resolver_con_rag_avanzado(state["pregunta"])

    return {
        "respuesta_RAG": resultado_rag["respuesta"],
        "citaciones": resultado_rag["citaciones"],
        "rag_exito": resultado_rag["rag_exito"],
        "accion_final": "RESOLUCIÓN_AUTOMÁTICA" if resultado_rag["rag_exito"] else "ESCALADO"
    }

# 4. Nodo Solicitar Detalles (Falta información técnica específica)
def nodo_solicitar_detalles(state: AgentState) -> AgentState:
    print("[NODO: Solicitar Detalles] Formulando plantilla de requerimientos...")
    motivo = state["triaje"].get("motivo", "Falta de información clave.")
    respuesta = (
        f"**ASISTENTE OCI**: Necesitamos datos adicionales para poder asesorarte.\n"
        f"**Razón**: {motivo}\n\n"
        f"Por favor, facilítanos la siguiente información si aplica:\n"
        f"- ID o Nombre del Pozo (ej. Pozo PCP-04)\n"
        f"- Tipo de mantenimiento / Herramienta involucrada\n"
        f"- ¿Se encuentra en zona de riesgo o bajo condiciones climáticas adversas?"
    )
    return {
        "respuesta_RAG": respuesta,
        "citaciones": [],
        "rag_exito": False,
        "accion_final": "SOLICITAR_DETALLES"
    }

# 5. Nodo Alertar Supervisor (Human-In-The-Loop)
def nodo_alertar_supervisor(state: AgentState) -> AgentState:
    print("[NODO: Alerta Supervisor] Ejecutando protocolo de seguridad HITL...")
    # Buscamos información en el RAG para dar una propuesta preliminar basada en datos
    resultado_rag = resolver_con_rag_avanzado(state["pregunta"])

    propuesta_datos = resultado_rag["respuesta"] if resultado_rag["rag_exito"] else "No se encontraron manuales específicos para este escenario crítico o la propuesta no superó los controles de seguridad de alucinación."

    alerta_hitl = (
        f"**ALERTA DE SEGURIDAD / ESCENARIO CRÍTICO DETECTADO** \n"
        f"**Urgencia**: {state['triaje'].get('urgency', 'ALTA')} | **Motivo**: {state['triaje'].get('motivo', 'Riesgo Técnico')}\n"
        f"----------------------------------------------------------------------\n"
        f"**Recomendación Asistida por IA (Basada en datos)**:\n"
        f"{propuesta_datos}\n"
        f"----------------------------------------------------------------------\n"
        f"**PROTOCOLO HUMAN-IN-THE-LOOP (HITL)**:\n"
        f"Esta consulta involucra operaciones con riesgos potenciales o decisiones de alta prioridad.\n"
        f"**Supervisor asignado**: Por favor evalúe el estado climático actual, el equipo de protección (EPP) y valide físicamente el procedimiento antes de autorizar la maniobra en campo.\n\n"
        f"**[ ] Aprobar Operación   |   [ ] Reconducir Protocolo   |   [ ] Solicitar Verificación de Campo**"
    )

    return {
        "respuesta_RAG": alerta_hitl,
        "citaciones": resultado_rag["citaciones"],
        "rag_exito": resultado_rag["rag_exito"],
        "accion_final": "ALERTA_SUPERVISOR_HITL"
    }


### 4.3 - Definicion de Aristas de Grapho

In [25]:
 # --- LÓGICA DE RUTAS (ARISTAS CONDICIONALES) ---

def arista_decision_triaje(state: AgentState) -> str:
    """Deriva el flujo en función del análisis de riesgo del triaje operativo."""
    decision = state["triaje"]["decision"]
    if decision == "AUTO_RESOLVER":
        return "ir_a_rag"
    elif decision == "PEDIR_INFO":
        return "ir_a_detalles"
    elif decision == "ALERTAR_SUPERVISOR":
        return "ir_a_alerta"
    raise ValueError(f"Decisión inválida: {decision}")

def arista_evaluacion_rag(state: AgentState) -> str:
    """Verifica si la consulta resuelta por RAG requiere escalación por falta de información o alucinación."""
    if state["rag_exito"]:
        return "finalizar"

    # Si falló la búsqueda en manuales o se detectó alucinación en un tema sensible, escalamos a supervisor
    palabras_criticas = ["riesgo", "accidente", "emergencia", "pulling", "tensión", "daño", "parada", "tormenta", "viento"]
    if any(p in state["pregunta"].lower() for p in palabras_criticas):
        print("Fallo en RAG o Alucinación detectada sobre tema de riesgo. Redirigiendo a Alerta de Supervisor.")
        return "ir_a_alerta"

    return "ir_a_detalles"

### 4.4 - Contruccion de Workflow de LangGraph

In [26]:
# --- CONSTRUCCIÓN DEL WORKFLOW DE LANGGRAPH ---
workflow_operativo = StateGraph(AgentState)

# Adición de Nodos de Acción
workflow_operativo.add_node("triaje", nodo_triaje_operaciones)
workflow_operativo.add_node("auto_resolver", nodo_auto_resolver_operaciones)
workflow_operativo.add_node("pedir_info", nodo_solicitar_detalles)
workflow_operativo.add_node("alertar_supervisor", nodo_alertar_supervisor)

# Ruta inicial
workflow_operativo.add_edge(START, "triaje")

# Conectores condicionales desde Triaje
workflow_operativo.add_conditional_edges(
    "triaje",
    arista_decision_triaje,
    {
        "ir_a_rag": "auto_resolver",
        "ir_a_detalles": "pedir_info",
        "ir_a_alerta": "alertar_supervisor"
    }
)

# Conectores condicionales desde Auto Resolver (RAG)
workflow_operativo.add_conditional_edges(
    "auto_resolver",
    arista_evaluacion_rag,
    {
        "finalizar": END,
        "ir_a_alerta": "alertar_supervisor",
        "ir_a_detalles": "pedir_info"
    }
)

# Los nodos finales apuntan directamente a la terminación de la ejecución
workflow_operativo.add_edge("pedir_info", END)
workflow_operativo.add_edge("alertar_supervisor", END)

# Compilación final del Grafo
grafo_operativo = workflow_operativo.compile()
print("¡Grafo de control operativo e HITL compilado con éxito!")


¡Grafo de control operativo e HITL compilado con éxito!


In [34]:
from langgraph.graph import START, END, StateGraph

try:
  graph_bytes = grafo_operativo.get_graph().draw_mermaid_png()
  with open("flujo_agente.png", "wb") as f:
      f.write(graph_bytes)
  print("Imagen del grafo guardada como 'flujo_agente.png' en esta carpeta.\n")
except Exception as e:
  print(f"No se pudo renderizar la imagen del grafo (requiere dependencias adicionales de Mermaid): {e}\n")

✅ Imagen del grafo guardada como 'flujo_agente.png' en esta carpeta.



## 5 - Espacio de pruebas operativas

In [41]:
from IPython.display import display, Markdown

# 1. Función para el renderizado estético de resultados (Pilar 4)
def mostrar_resultado_formateado(resultado: Dict):
    """Genera un reporte técnico estructurado, limpio y plano en consola."""
    pregunta = resultado.get("pregunta", "")
    triaje = resultado.get("triaje", {})
    accion = resultado.get("accion_final", "N/A")
    respuesta = resultado.get("respuesta_RAG", "")
    citaciones = resultado.get("citaciones", [])

    # Construimos el reporte usando saltos de línea limpios sin sangrías ocultas
    reporte = (
        f"--- 📊 RESULTADO DEL TRIAJE ---\n"
        f"Consulta: \"{pregunta}\"\n"
        f"Decisión: {triaje.get('decision', 'N/A')}\n"
        f"Urgencia: {triaje.get('urgency', 'N/A')}\n"
        f"Motivo: {triaje.get('motivo', 'N/A')}\n"
        f"Acción del Grafo: {accion}\n\n"
        f"--- 🤖 RESPUESTA Y DIRECTRICES TÉCNICAS ---\n"
        f"{respuesta}\n"
    )

    # Si existen citaciones, se añaden en formato plano
    if citaciones:
        reporte += "\n--- 📂 CITACIONES DE RESPALDO ---\n"
        for i, cit in enumerate(citaciones, 1):
            # Soporta tanto string directo como objetos/diccionarios de LangChain/LlamaIndex
            if isinstance(cit, dict):
                fuente = cit.get("metadata", {}).get("file_path", str(cit))
            elif hasattr(cit, "metadata"):
                fuente = cit.metadata.get("file_path", str(cit))
            else:
                fuente = str(cit)

            reporte += f"  [{i}] Fuente: '{fuente}'\n"

    reporte += "\n--- ⚙️ OCI Agent RAG v2.5 ---"

    # IMPRIMIR EN TEXTO PLANO (Evita que Colab renderice Markdown o HTML complejo)
    print(reporte)

# 2. Ejecución de casos de prueba integrados con el pipeline de seguridad
casos_prueba_operativa = [
    # Escenario crítico -> Debe levantar alerta y dar reporte estructurado HITL
    "¿Que pozos tienen tareas de Mantenimiento estan porgramadas?",

    # Consulta estándar de bajo riesgo -> Debe auto-resolver citando las fuentes
    "¿Cuando es la intervencion de pulling del pozo P-404?"
]

for idx, consulta in enumerate(casos_prueba_operativa, 1):
    print(f"\nEJECUTANDO TEST OPERATIVO #{idx}...")
    resultado_final = grafo_operativo.invoke({"pregunta": consulta})

    # Despliegue visual en el formato final
    mostrar_resultado_formateado(resultado_final)


EJECUTANDO TEST OPERATIVO #1...
[NODO: Triaje] Analizando riesgos de la consulta...
[NODO: Auto Resolver] Consultando manuales técnicos...
[GUARDRAIL] Alucinación detectada: La 'Respuesta Propuesta' incurre en una alucinación en la última frase: 'Recuerde que el Supervisor de Campo tiene la decisión final frente a cualquier imprevisto técnico o climático.'. Aunque el 'Contexto Técnico' (RECURSO 2) es un 'Procedimiento_supervisor_de_campo_OCI.pdf' y detalla reglas de priorización y asignación de recursos, no establece explícitamente que el Supervisor de Campo tenga la 'decisión final frente a cualquier imprevisto técnico o climático'. El documento describe un flujo lógico y una matriz de decisión, pero no otorga una autoridad tan amplia y generalizada de manera explícita en el texto proporcionado. El resto de la información sobre los pozos y las advertencias de seguridad está directamente extraída del RECURSO 1 y RECURSO 2, siendo fiel al contexto.
[NODO: Solicitar Detalles] Formulando